# RAG (Retrieval-Augmented Generator)

## Pinecone integration
## 1. Prerequisites

To build a RAG system, a specialized stack is required. langchain-pinecone and pinecone-client are installed to manage the vector database, and langchain-google-genai is installed to access Gemini's embedding and chat functions. These tools enable the connection of static documents with generative AI.

In [ ]:
pip install -U langchain-google-genai langchain-pinecone pinecone-client langchain-community

In [ ]:
pip install beautifulsoup4 langgraph langchain python-dotenv

## 2. Setup and credentials

Here, python-dotenv is used to load the API keys for both Pinecone and Google. Initializing the Pinecone client at this stage establishes a secure connection to the cloud infrastructure.

In [ ]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv() # Load keys from .env

pinecone_api_key = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

## 3. Initialization

Before storing the data, the space it will occupy is defined. A Pinecone index with 768 dimensions is created to meet Gemini's embedding requirements. The GoogleGenerativeAIembeddings model, responsible for translating human language into the mathematical vectors that Pinecone understands, is also initialized.

In [ ]:
from pinecone import ServerlessSpec
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_pinecone import PineconeVectorStore

index_name = "langchain-test-index"

# Create Index
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=768,  # Change from 1536 to 768 for Gemini
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

# Setup Google Embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    output_dimensionality=768)

# Initialize vector store
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

## 4. Manage vector store (Add items)

This section demonstrates the basic CRUD operations of a vector database. A set of sample documents is created, assigned unique identifiers (UUIDs), and uploaded to Pinecone. It also shows how to delete specific entries, which is essential for keeping a production knowledge base up to date.

In [ ]:
from uuid import uuid4
from langchain_core.documents import Document

documents = [
    Document(page_content="I had chocolate chip pancakes and scrambled eggs for breakfast.", metadata={"source": "tweet"}),
    Document(page_content="The weather forecast for tomorrow is cloudy.", metadata={"source": "news"}),
    Document(page_content="Building an exciting new project with LangChain!", metadata={"source": "tweet"}),
    Document(page_content="Robbers broke into the city bank and stole $1 million.", metadata={"source": "news"}),
    Document(page_content="Wow! That was an amazing movie.", metadata={"source": "tweet"}),
    Document(page_content="Is the new iPhone worth the price?", metadata={"source": "website"}),
    Document(page_content="The top 10 soccer players in the world.", metadata={"source": "website"}),
    Document(page_content="LangGraph is the best framework for agentic apps!", metadata={"source": "tweet"}),
    Document(page_content="The stock market is down 500 points today.", metadata={"source": "news"}),
    Document(page_content="I have a bad feeling I am going to get deleted :(", metadata={"source": "tweet"}),
]

uuids = [str(uuid4()) for _ in range(len(documents))]
vector_store.add_documents(documents=documents, ids=uuids)

vector_store.delete(ids=[uuids[-1]])

## 5. Query vector store

This is a test of semantic search. Unlike traditional keyword search, similarity search looks for meaning. Even if the query doesn't use the exact same words as the stored documents, Pinecone can find the most contextually relevant information by calculating the distance between vectors in a high-dimensional space.

In [10]:
print("--- Query Directly ---")
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

print("\n--- Similarity Search with Score ---")
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

--- Query Directly ---
* Building an exciting new project with LangChain! [{'source': 'tweet'}]
* LangGraph is the best framework for agentic apps! [{'source': 'tweet'}]

--- Similarity Search with Score ---
* [SIM=0.707353] The weather forecast for tomorrow is cloudy. [{'source': 'news'}]


The number `SIM=0.7073` indicates how close the vectors are.

## Build a RAG
## 1. Indexing (Load, Split, and Store)

This is the "Data Ingestion" phase of the RAG. Content is extracted from an active technical blog using `WebBaseLoader`. Since LLMs have limited context windows, `RecursiveCharacterTextSplitter` is used to split the long article into 1000-character fragments. These fragments are vectorized and stored in Pinecone, forming the searchable knowledge base.

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load: Only keep post title, headers, and content
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

# Split: break large Documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)

# Store: Index chunks in Pinecone
document_ids = vector_store.add_documents(documents=all_splits)

print(f"Split blog post into {len(all_splits)} sub-documents.")
print(f"First 3 document IDs: {document_ids[:3]}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Split blog post into 63 sub-documents.
First 3 document IDs: ['fe798bd4-c8b7-4872-aeb7-db47f58628f2', '85a836be-11e7-4a2b-b839-d6362c7ef0a3', '0268bfc6-4b89-4bfd-9ca9-ab92f698e4e7']


## 2. Retrieval the agent

Here, a custom tool called `retrieve_context` is defined. This function acts as a bridge: it takes a query, searches our Pinecone index, and returns the most relevant text snippets along with their metadata. By decorating it with `@tool`, it becomes visible so the AI ​​agent can use it when it needs more information.

In [ ]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

# Construct a tool for retrieving context
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

## 3. Generation the agent

Now the "brain" is assembled. `ChatGoogleGenerativeAI` (Gemini) is configured and provided with the retrieval tool. The system message is crucial: it tells the LLM that they have a "book" (the tool) they can consult to answer questions accurately, rather than guessing or imagining.

In [ ]:
from langchain.agents import create_agent

tools = [retrieve_context]

# Setup the LLM (Gemini)
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Define the system prompt
prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)

# Create the Agent
agent = create_agent(model, tools, system_prompt=prompt)

## 4. Execution

In this final stage, the agent runs. Observe the output of `agent.stream`: you can see the agent's thought process. First, it realizes it doesn't have the answer in memory, then it calls the `retrieve_context` tool, reads the blog content, and finally synthesizes an informed, evidence-based response for the user.

In [19]:
query = (
    "What is the standard method for Task Decomposition?\n\n"
    "Once you get the answer, look up common extensions of that method."
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the standard method for Task Decomposition?

Once you get the answer, look up common extensions of that method.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (f88fb33d-a849-41b7-ac6e-e5e5b2550046)
 Call ID: f88fb33d-a849-41b7-ac6e-e5e5b2550046
  Args:
    query: standard method for Task Decomposition
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 2578.0}
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on

The Retrieval mechanism acts as an "external memory" that supplies specific data to the generative model in real time. When a query is made, the system searches Pinecone for the most relevant text fragments using vector similarity. These fragments are injected directly into the prompt received by the LLM (Gemini), transforming an open-ended question into a "reading comprehension" task. In this way, the model does not rely solely on its prior training but uses the retrieved context to support its response.

The integration with the generative model enhances accuracy and reduces "hallucinations," as the model now prioritizes evidence from the provided context over its own general knowledge. By processing this contextual information along with the original question, the LLM can synthesize much more detailed and up to date responses on topics it was initially unfamiliar with, such as the specific content of a private blog or recent technical documents. In the case of this agent, this integration is dynamic: the model decides when it needs to query the database and how to use that data to enrich its final explanation.